# Ekspor Eval-Only Gelombang-4 — Data Non-Papua BARU untuk Val/Test `improve_model.ipynb`

## Konteks
`improve_model.ipynb` (fine-tuning model_1) menunjukkan IoU 5 kelas minoritas yang rendah dan
liar berfluktuasi antar-epoch. Akar masalah: Val/Test Papua-only holdout punya piksel kelas
minoritas yang SANGAT sedikit (Tambang val cuma 0,0028% dari 1,27 miliar piksel) -> estimasi
IoU kelas itu secara statistik tidak reliable.

**Keputusan (USER, sengaja override `docs/model/RENCANA_EKSEKUSI_LANJUTAN.md` §1.4/§3 yang
sebelumnya melarang ini):** masukkan data non-Papua juga ke Val/Test.

**Kendala krusial yang menentukan desain notebook ini:**
1. **Leakage** — model_1 sudah dilatih 80 epoch dengan **100% data transfer yang SUDAH ADA**
   (`ForestWatch_Patches_Transfer/<slug>/`, dikonfirmasi nyata: Tambang 42,9jt px, Permukiman
   51,2jt, Sawit 89,7jt, Lahan Terbuka 69,1jt, Pertanian Lain 214jt). Memindahkan sebagian data
   ITU ke Val/Test = mengevaluasi model dengan data yang sudah dihafalnya.
2. Split Papua (`Bahan_Training_Fix/val.tar`/`test.tar`) sudah **dibekukan** sbg acuan
   `metrics.json` model_1/2/3 — TIDAK disentuh sama sekali oleh notebook ini.

**Solusi:** ekspor region **BARU SEPENUHNYA** (belum pernah dipakai di gelombang 1-3 manapun),
ke folder Drive **terpisah** (`ForestWatch_Patches_TransferEval/`) — data ini otomatis aman dari
leakage karena belum pernah dilihat model manapun. Split akhir notebook ini menghasilkan
train_new/val_new/test_new (manifest) — tapi **`improve_model.ipynb` HANYA mengambil
val_new+test_new**, `train_new` sengaja tidak dipakai utk fine-tuning model_1 (tersimpan sbg
aset utk skenario retrain-dari-nol di masa depan).

**Yang TIDAK disentuh:** `forestwatch_papua_full_pipeline.ipynb`, `train_model_1/2/3.ipynb`,
`compare_and_select_best_model.ipynb`, `Bahan_Training_Fix` — semua tetap seperti sekarang.


In [ ]:
# === COLAB SETUP (clone pertama kali / PULL sesi berikutnya) ===
%cd /content
!git -C fw_repo pull -q || git clone --depth 1 https://github.com/Ridho-Dwi-Syahputra/forestwatch-model.git fw_repo
%cd fw_repo
!pip install -q -e ".[gee,gis,ml]"

import sys, importlib
if '/content/fw_repo/model/src' not in sys.path:
    sys.path.insert(0, '/content/fw_repo/model/src')
for mod in list(sys.modules.keys()):
    if mod == 'forestwatch' or mod.startswith('forestwatch.'):
        del sys.modules[mod]
importlib.invalidate_caches()

import forestwatch
print(f'forestwatch v{forestwatch.__version__}')


In [ ]:
# === MOUNT GOOGLE DRIVE ===
from google.colab import drive
drive.mount('/content/drive')
print('Drive ter-mount.')


In [ ]:
# === KONFIGURASI PATH (folder BARU, terpisah dari ForestWatch_Patches_Transfer) ===
from pathlib import Path
from forestwatch.utils.io import save_json, load_json
from forestwatch.config import load_config

DRIVE_ROOT = Path('/content/drive/MyDrive/Satria Data 3.0')
TILES_TRANSFER_EVAL  = DRIVE_ROOT / 'ForestWatch_Tiles_TransferEval'
PATCHES_TRANSFER_EVAL = DRIVE_ROOT / 'ForestWatch_Patches_TransferEval'
DIST_DIR = DRIVE_ROOT / 'Distribution_Reports'
EDA_SAVE_DIR = DRIVE_ROOT / 'EDA_Visualizations'

for d in (TILES_TRANSFER_EVAL, PATCHES_TRANSFER_EVAL, DIST_DIR, EDA_SAVE_DIR):
    d.mkdir(parents=True, exist_ok=True)

cfg = load_config()
print('Folder BARU (eval-only, terpisah dari ForestWatch_Patches_Transfer):')
print(' -', TILES_TRANSFER_EVAL)
print(' -', PATCHES_TRANSFER_EVAL)


In [ ]:
# === AUTH GEE + Periode T2 ===
import ee
from forestwatch.gee.auth import init_ee

init_ee(project=cfg['project']['gee_project_id'])
T2 = cfg['periods']['t2']
print(f'GEE siap. Periode label T2={T2}.')


In [ ]:
# === Bagian E1 — EVAL_REGIONS (region BARU) + cross-check GANDA vs region yang SUDAH dipakai ===
# Cross-check (1) nama region, (2) OVERLAP KOORDINAT BBox -- nama beda tidak cukup, dua region
# bisa secara geografis tumpang-tindih meski namanya berbeda. Daftar di bawah = hasil grep PENUH
# seluruh sel region di forestwatch_papua_full_pipeline.ipynb (TRANSFER_REGIONS cell 48,
# TAMBANG_EXTRA_REGIONS cell 66, TAMBANG_EXTRA2_REGIONS cell 70, PERMUKIMAN_EXTRA_REGIONS cell 74,
# GEL3_REGIONS cell 78) -- LINTAS SEMUA KELAS (bbox eval kelas A bisa overlap bbox terpakai kelas
# B secara geografis, walau folder/slug-nya beda).
ALREADY_USED_BBOXES = {
    # --- Tambang ---
    'kaltim_sangatta':        (117.31,   0.31, 117.79,   0.79),
    'kaltim_sangatta_gel3':   (117.40,   0.25, 117.90,   0.75),
    'kalsel_tanahbumbu':      (115.41,  -3.69, 115.89,  -3.21),
    'sulteng_morowali':       (121.946, -2.974, 122.234, -2.686),
    'sumbawa_batuhijau':      (116.729, -9.062, 117.001, -8.838),
    'chile_chuquicamata':     (-68.996, -22.416, -68.804, -22.224),
    'chile_escondida':        (-69.156, -24.369, -68.964, -24.161),
    'usa_bingham':            (-112.236, 40.440, -112.044, 40.600),
    'aus_kalgoorlie':         (121.414, -30.850, 121.606, -30.690),
    'aus_huntervalley':       (150.725, -32.686, 151.125, -32.334),
    'australia_huntervalley_gel3': (150.70, -32.75, 151.20, -32.25),
    'ger_hambach':            (6.412,   50.827,   6.668,  51.003),
    'halmahera_wedabay':      (127.62,  -0.75, 128.32,  -0.05),
    'aus_pilbara':            (119.35, -23.75, 120.15, -22.95),
    'usa_powderriver':        (-105.95,  43.85, -105.05,  44.75),
    'kalsel_adaro':           (115.10,  -2.50, 115.80,  -1.80),
    'brazil_carajas':         (-50.60,  -6.40, -49.90,  -5.70),
    'brazil_carajas_gel3':    (-50.40,  -6.20, -50.00,  -5.80),
    'papua_grasberg':         (137.00,  -4.30, 137.25,  -4.00),
    'peru_antamina':          (-77.15,  -9.62, -76.95,  -9.42),
    'safrica_witbank':        (29.10,  -26.00,  29.40,  -25.75),
    'sumsel_tanjungenim':     (103.65,  -3.90, 104.10,  -3.45),
    'safrica_sishen':         (22.70,  -28.10,  23.20,  -27.60),
    'canada_athabasca':       (-111.75,  56.95, -111.30,  57.40),
    'australia_bowenbasin':   (148.00, -22.00, 148.50, -21.50),
    # --- Permukiman ---
    'jabodetabek': (106.70, -6.35, 106.95, -6.10), 'bandung': (107.55, -6.97, 107.70, -6.85),
    'surabaya': (112.68, -7.32, 112.82, -7.20), 'medan': (98.62, 3.52, 98.74, 3.64),
    'makassar': (119.40, -5.18, 119.52, -5.08), 'palembang': (104.62, -3.08, 104.88, -2.85),
    'semarang': (110.32, -7.05, 110.52, -6.90), 'denpasar': (115.13, -8.75, 115.32, -8.55),
    'balikpapan': (116.78, -1.32, 116.98, -1.12), 'pekanbaru': (101.35, 0.42, 101.58, 0.62),
    'papua_jayapura': (140.66, -2.62, 140.78, -2.50), 'papua_merauke': (140.36, -8.52, 140.48, -8.40),
    'papua_timika': (136.84, -4.58, 136.96, -4.46), 'papua_sorong': (131.22, -0.92, 131.34, -0.80),
    'papua_biak': (136.04, -1.20, 136.16, -1.08), 'maluku_ambon': (128.14, -3.72, 128.26, -3.60),
    'ntt_kupang': (123.54, -10.22, 123.70, -10.08), 'bogor_depok': (106.65, -6.75, 107.05, -6.35),
    'padang': (100.20, -1.15, 100.60, -0.75), 'banjarmasin': (114.40, -3.50, 114.80, -3.10),
    'pontianak': (109.10, -0.20, 109.50, 0.20), 'manado': (124.65, 1.30, 125.05, 1.70),
    'yogyakarta': (110.25, -8.00, 110.60, -7.65),
    # --- Sawit ---
    'riau_pelalawan': (101.40, 0.20, 101.70, 0.50), 'sumut': (99.50, 2.00, 99.80, 2.30),
    'kalbar': (109.50, 0.00, 109.80, 0.30), 'kalteng': (112.50, -2.20, 112.80, -1.90),
    'sumsel_musibanyuasin': (103.80, -2.85, 104.20, -2.45),
    'riau_rokanhilir': (100.90, 1.50, 101.30, 1.90), 'riau_kampar': (101.00, 0.00, 101.40, 0.40),
    'riau_indragirihilir': (102.80, -0.70, 103.20, -0.30),
    'sumut_labuhanbatu': (99.90, 1.85, 100.30, 2.25), 'kalbar_ketapang': (110.20, -1.80, 110.60, -1.40),
    # --- Lahan Terbuka ---
    'kaltim_tepitambang': (117.30, 0.30, 117.55, 0.55), 'kalteng_pascabakar': (113.50, -2.50, 113.80, -2.20),
    # --- Pertanian Lain ---
    'pantura_indramayu': (108.20, -6.45, 108.45, -6.25), 'lampung_ladang': (105.20, -5.10, 105.45, -4.90),
}

ALREADY_USED = {
    'tambang': {
        'kaltim_sangatta', 'kalsel_tanahbumbu', 'sulteng_morowali', 'sumbawa_batuhijau',
        'chile_chuquicamata', 'chile_escondida', 'usa_bingham', 'aus_kalgoorlie',
        'aus_huntervalley', 'ger_hambach', 'halmahera_wedabay', 'aus_pilbara',
        'usa_powderriver', 'kalsel_adaro', 'brazil_carajas', 'papua_grasberg',
        'peru_antamina', 'safrica_witbank', 'sumsel_tanjungenim', 'safrica_sishen',
        'canada_athabasca', 'australia_bowenbasin', 'australia_huntervalley', 'kaltim_paser',
    },
    'permukiman': {
        'jabodetabek', 'bandung', 'surabaya', 'medan', 'makassar', 'palembang', 'semarang',
        'denpasar', 'balikpapan', 'pekanbaru', 'papua_jayapura', 'papua_merauke',
        'papua_timika', 'papua_sorong', 'papua_biak', 'maluku_ambon', 'ntt_kupang',
        'bogor_depok', 'padang', 'banjarmasin', 'pontianak', 'manado', 'yogyakarta',
    },
    'sawit': {
        'riau_pelalawan', 'sumut', 'kalbar', 'kalteng', 'sumsel_musibanyuasin',
        'riau_rokanhilir', 'riau_kampar', 'riau_indragirihilir', 'sumut_labuhanbatu',
        'kalbar_ketapang', 'jambi_tebo', 'kalsel_kotabaru',
    },
    'lahan_terbuka': {'kaltim_tepitambang', 'kalteng_pascabakar'},
    'pertanian_lain': {'pantura_indramayu', 'lampung_ladang'},
}

# Kandidat region gelombang-4 (eval-only) -- bbox APPROX, diverifikasi via preview+GATE (E2)
# di bawah sebelum submit export (E3). Dipilih genuinely BARU (provinsi/negara/morfologi
# belum dipakai di gelombang manapun).
EVAL_REGIONS = {
    'tambang': [
        ('bangka_timah',        (106.10,  -2.15, 106.40,  -1.85)),  # tambang timah/tin Bangka
        ('philippines_surigao', (125.45,   9.55, 125.75,   9.85)),  # nikel laterit
        ('mongolia_oyutolgoi',  (106.70,  42.90, 107.00,  43.15)),  # tembaga-emas
        ('india_bailadila',     ( 81.15,  18.55,  81.45,  18.85)),  # bijih besi
    ],
    'permukiman': [
        ('cirebon',      (108.52,  -6.78, 108.62,  -6.68)),
        ('malang',       (112.58,  -7.99, 112.68,  -7.92)),
        ('tasikmalaya',  (108.18,  -7.38, 108.28,  -7.30)),
        ('manokwari',    (134.04,  -0.90, 134.12,  -0.83)),  # Papua Barat, IN-DOMAIN
    ],
    'sawit': [
        ('aceh_acehtimur', (97.65,   4.40,  97.95,   4.70)),
        ('sulbar_mamuju',  (119.00,  -2.75, 119.30,  -2.45)),
    ],
    'lahan_terbuka': [
        ('riau_tepigambut',      (102.30,  0.10, 102.60,  0.40)),
        ('ntb_sumbawa_pascabakar', (117.40, -8.65, 117.70, -8.35)),
    ],
    'pertanian_lain': [
        ('jatim_ladang', (112.30, -7.55, 112.60, -7.25)),
        ('bali_sawah',   (115.15, -8.45, 115.40, -8.20)),
    ],
}


def _bbox_overlap(a, b):
    """True bila 2 bbox (lon0,lat0,lon1,lat1) berpotongan (area > 0)."""
    ax0, ay0, ax1, ay1 = a
    bx0, by0, bx1, by1 = b
    return ax0 < bx1 and bx0 < ax1 and ay0 < by1 and by0 < ay1


print('=== Cross-check 1: nama region vs ALREADY_USED (gelombang 1-3) ===')
collisions = []
for slug, regions in EVAL_REGIONS.items():
    for name, bbox in regions:
        if name in ALREADY_USED.get(slug, set()):
            collisions.append((slug, name))
assert not collisions, f'Nama region COLLIDE dengan yang sudah dipakai: {collisions}'
print('OK -- tidak ada nama region yang sama dengan gelombang 1-3.')

print('\n=== Cross-check 2: OVERLAP KOORDINAT bbox vs SEMUA bbox terpakai (lintas kelas) ===')
geo_overlaps = []
for slug, regions in EVAL_REGIONS.items():
    for name, bbox in regions:
        for used_name, used_bbox in ALREADY_USED_BBOXES.items():
            if _bbox_overlap(bbox, used_bbox):
                geo_overlaps.append((slug, name, used_name))
assert not geo_overlaps, (
    f'Region BARU overlap geografis dgn region yg sudah dipakai (leakage risk): {geo_overlaps}')
print(f'OK -- {sum(len(r) for r in EVAL_REGIONS.values())} region baru TIDAK overlap koordinat '
      f'dgn {len(ALREADY_USED_BBOXES)} bbox yang sudah pernah diekspor (gelombang 1-3).')

for slug, regions in EVAL_REGIONS.items():
    print(f'  {slug:<16}: {len(regions)} region baru -> {[n for n, _ in regions]}')


In [ ]:
# === Bagian E2 — Preview + GATE: pastikan region BARU memang mengandung kelas targetnya ===
# Sama spt pola GATE 12.1b/12.9b: reduceRegion(frequencyHistogram) per region -> % piksel
# kelas target. Region <1% di-flag (peringatan, tidak otomatis dibuang -- review manual dulu).
import numpy as np
from forestwatch.gee.composite import s2_composite
from forestwatch.gee.label_fusion import build_label
from forestwatch.constants import CLASS_NAMES, CLASS_SLUGS, N_CLASSES, DEG_TO_M

EST_SCALE = 100
MIN_OWN_FRAC = 0.01
SLUG_TO_CLASS = {v: k for k, v in enumerate(CLASS_SLUGS)}
TARGET_ADD_PER_CLASS = {2: 100_000_000, 3: 100_000_000, 4: 100_000_000, 5: 50_000_000, 6: 100_000_000}


def _bbox_area_px(bbox, scale_m=10):
    x0, y0, x1, y1 = bbox
    w_m = (x1 - x0) * DEG_TO_M * np.cos(np.radians((y0 + y1) / 2))
    h_m = (y1 - y0) * DEG_TO_M
    return (w_m / scale_m) * (h_m / scale_m)


print(f'GATE pre-export gelombang-4 (scale {EST_SCALE} m)...\n')
region_reports = []
gate = {}
# Estimasi piksel SEMUA kelas (bukan cuma kelas target tiap region) -- utk tabel agregat di
# bawah, gaya yg sama persis dgn output distribusi gelombang lama (cell 82 pipeline notebook).
agg_est_px = {c: 0.0 for c in range(N_CLASSES)}
for slug, regions in EVAL_REGIONS.items():
    own_cls = SLUG_TO_CLASS[slug]
    for name, bbox in regions:
        box = ee.Geometry.Rectangle(list(bbox))
        img = s2_composite(T2, box)
        label = build_label(box, T2, composite=img)
        fh = label.reduceRegion(ee.Reducer.frequencyHistogram(), box, EST_SCALE,
                                 maxPixels=int(1e13)).getInfo().get('label', {})
        cnt = {c: float(fh.get(str(c), fh.get(f'{c}.0', 0)) or 0) for c in range(N_CLASSES)}
        tot = sum(cnt.values()) or 1.0
        own_frac = cnt[own_cls] / tot
        area_px = _bbox_area_px(bbox)
        region_reports.append({
            'slug': slug, 'name': name, 'area_px_native': area_px,
            'own_class': CLASS_NAMES[own_cls], 'own_class_frac': own_frac,
            'own_class_px_est': area_px * own_frac,
        })
        key = f'{slug}/{name}: >={MIN_OWN_FRAC * 100:.0f}% piksel {CLASS_NAMES[own_cls]}'
        gate[key] = own_frac >= MIN_OWN_FRAC
        print(f'  {slug:<16}/{name:<24}: {CLASS_NAMES[own_cls]:<14} frac={own_frac * 100:5.1f}% '
              f'(~{area_px * own_frac / 1e6:5.2f} jt px)')
        for c in range(N_CLASSES):
            agg_est_px[c] += area_px * (cnt[c] / tot)

print()
print('=== GATE E2: kelayakan region (memang ada piksel kelas targetnya) ===')
for k, v in gate.items():
    print(f"  [{'OK ' if v else 'X  '}] {k}")

# --- Output yang diminta: tabel agregat per-kelas vs target, gaya sama dgn cell distribusi lama ---
print()
print('Distribusi piksel GELOMBANG-4 (gabungan, ESTIMASI pre-export) vs target per-kelas '
      '(100 jt / Tambang 50 jt):')
for c in range(N_CLASSES):
    tgt = TARGET_ADD_PER_CLASS.get(c)
    pct = f'{100 * agg_est_px[c] / tgt:5.1f}%' if tgt else ' n/a '
    print(f'  {CLASS_NAMES[c]:<16}: {agg_est_px[c]:>12,.0f} px  ({pct:>6} target)')
print('\nCatatan: ini ESTIMASI pre-export (scale 100m, ekstrapolasi BBox) -- angka AKTUAL baru')
print('diketahui pasca-cut di E7. Memastikan kelima kelas (2-6) > 0% sebelum lanjut ke E3.')
for c in (2, 3, 4, 5, 6):
    assert agg_est_px[c] > 0, f'{CLASS_NAMES[c]} estimasi 0 px -- region gagal, revisi E1.'

save_json({'scale_m': EST_SCALE, 'min_own_frac': MIN_OWN_FRAC, 'regions': region_reports,
           'gate': gate, 'passed': all(gate.values()),
           'agg_estimate_px': {CLASS_NAMES[c]: agg_est_px[c] for c in range(N_CLASSES)}},
          DIST_DIR / 'pixel_estimate_gel4_eval.json')

n_weak = sum(1 for v in gate.values() if not v)
if n_weak:
    print(f'\n*** PERINGATAN: {n_weak} region <1% kelas targetnya -- review/ganti bbox di E1')
    print('    SEBELUM lanjut ke E3 (export). ***')
else:
    print('\nGATE LULUS -- semua region gelombang-4 relevan dgn kelas targetnya. Lanjut ke E3.')


In [ ]:
# === Bagian E3 — Export GEE (additif, folder BARU eval_<slug>) ===
assert all(_v for _v in __import__('json').load(open(DIST_DIR / 'pixel_estimate_gel4_eval.json'))['gate'].values()), (
    'GATE E2 belum lulus semua -- revisi region di E1 dulu (lihat [X] di output E2).')
from forestwatch.gee.tiles import make_tiles
from forestwatch.gee.export import export_tiles_grid

NX, NY = 2, 2
eval_tasks = []
for slug, regions in EVAL_REGIONS.items():
    for name, bbox in regions:
        box = ee.Geometry.Rectangle(list(bbox))
        img = s2_composite(T2, box)
        label = build_label(box, T2, composite=img)
        stack = img.addBands(label.toFloat())
        tiles = make_tiles(box, nx=NX, ny=NY)
        tasks = export_tiles_grid(
            stack, tiles,
            name_prefix=f'eval_{slug}_{name}_tile',
            folder=f'eval_{slug}',
            scale=cfg['sentinel2']['scale'],
            max_pixels=int(cfg['export']['max_pixels']),
        )
        eval_tasks.extend(tasks)

print(f'{len(eval_tasks)} task ekspor gelombang-4 (eval-only) disubmit -> folder Drive eval_<slug>/.')
print('Pantau: https://code.earthengine.google.com/tasks')
print('Tunggu SEMUA task COMPLETED sebelum lanjut ke E4 (pindah tile nyasar + cut patches).')


In [ ]:
# === Bagian E4 — Pindahkan tile yang nyasar (pola identik Bagian 12.3b pipeline notebook) ===
import shutil
for slug in EVAL_REGIONS:
    src = DRIVE_ROOT / 'Augmented_Patches' / f'eval_{slug}'
    dst = TILES_TRANSFER_EVAL / slug
    dst.mkdir(parents=True, exist_ok=True)
    if src.exists():
        moved = 0
        for f in src.glob('*.tif'):
            shutil.move(str(f), str(dst / f.name))
            moved += 1
        if moved:
            print(f'{slug}: {moved} file dipindah -> {dst}')
print('Selesai cek tile nyasar.')


In [ ]:
# === Bagian E5 — cut_patches_resilient (disalin verbatim dari Bagian 12.4 pipeline notebook) ===
import numpy as np
import rasterio
import shutil
import time
from rasterio.windows import Window
from pathlib import Path
from tqdm.auto import tqdm


def _remount_drive():
    from google.colab import drive
    try:
        drive.flush_and_unmount()
    except Exception:
        pass
    time.sleep(3)
    drive.mount('/content/drive', force_remount=True)
    time.sleep(2)
    print("  -> Drive di-remount.")


def _read_done_count(done_marker):
    try:
        if done_marker.exists():
            return int(done_marker.read_text().strip())
    except (OSError, ValueError):
        return None
    return None


def cut_patches_resilient(tile_dir, patch_base_dir, *,
                          patch_size=256, stride=256, max_nan_ratio=0.3,
                          n_channels_image=6, local_tmp='/content/_tmp_patches'):
    tile_files = sorted(Path(tile_dir).glob('*.tif'))
    if not tile_files:
        raise FileNotFoundError(f"Tidak ada .tif di {tile_dir}")

    patch_base = Path(patch_base_dir)
    local_root = Path(local_tmp)
    n_tiles = len(tile_files)
    total = 0

    for ti, tif in enumerate(tile_files):
        drive_out = patch_base / f'tile_{ti:03d}'
        done_marker = drive_out / '_DONE'
        header = f"Tile {ti+1}/{n_tiles}  (tile_{ti:03d})"

        try:
            done_n = _read_done_count(done_marker)
            actual_n = len(list(drive_out.glob('p*.npz'))) if drive_out.exists() else 0
        except OSError:
            _remount_drive()
            done_n = _read_done_count(done_marker)
            actual_n = len(list(drive_out.glob('p*.npz'))) if drive_out.exists() else 0

        if done_n is not None and actual_n == done_n:
            total += actual_n
            tag = "laut/kosong" if done_n == 0 else f"{actual_n} patch"
            print(f"{header}: SKIP - komplit ({tag})")
            continue

        try:
            if drive_out.exists():
                shutil.rmtree(drive_out, ignore_errors=True)
        except OSError:
            _remount_drive()
            shutil.rmtree(drive_out, ignore_errors=True)

        ltile = local_root / f'tile_{ti:03d}'
        if ltile.exists():
            shutil.rmtree(ltile)
        ltile.mkdir(parents=True, exist_ok=True)

        idx = 0
        with rasterio.open(tif) as src:
            W, H, nb = src.width, src.height, src.count
            row_list = list(range(0, H - patch_size + 1, stride))
            col_list = list(range(0, W - patch_size + 1, stride))
            pbar = tqdm(total=len(row_list) * len(col_list), desc=header, unit="win", leave=True)
            for r in row_list:
                for c in col_list:
                    arr = src.read(window=Window(c, r, patch_size, patch_size))
                    pbar.update(1)
                    if arr.shape != (nb, patch_size, patch_size):
                        continue
                    img = arr[:n_channels_image].astype('float32')
                    if np.isnan(img).mean() > max_nan_ratio:
                        continue
                    img = np.nan_to_num(img)
                    kw = dict(img=img, tile=tif.name, row=r, col=c)
                    if nb > n_channels_image:
                        kw['lab'] = np.nan_to_num(arr[n_channels_image], nan=0.0).astype('uint8')
                    np.savez_compressed(ltile / f'p{idx:05d}.npz', **kw)
                    idx += 1
                    pbar.set_postfix(patch=idx)
            pbar.close()

        local_files = sorted(ltile.glob('p*.npz'))
        assert len(local_files) == idx, "jumlah file lokal tidak konsisten"

        synced = False
        for attempt in range(1, 8):
            try:
                drive_out.mkdir(parents=True, exist_ok=True)
                for f in local_files:
                    dst = drive_out / f.name
                    if (not dst.exists()) or (dst.stat().st_size != f.stat().st_size):
                        shutil.copy2(f, dst)
                drive_n = len(list(drive_out.glob('p*.npz')))
                if drive_n == idx:
                    done_marker.write_text(str(idx))
                    synced = True
                    break
                print(f"  verifikasi belum cocok: Drive={drive_n} vs lokal={idx} (attempt {attempt})")
            except OSError as e:
                print(f"  sync gagal (attempt {attempt}): {e}")
                _remount_drive()
            time.sleep(2)
        shutil.rmtree(ltile, ignore_errors=True)

        if not synced:
            print(f"{header}: GAGAL verifikasi sync - STOP. Hapus folder tile ini di Drive lalu jalankan ulang.")
            return total

        total += idx
        tag = "LAUT/kosong (0 patch)" if idx == 0 else f"{idx} patch"
        print(f"{header}: SELESAI - {tag} -> Drive/tile_{ti:03d}/\n")

    print(f"=== SEMUA TILE SELESAI. Total {total} patch ===")
    return total


print("cut_patches_resilient siap.")


In [ ]:
# === Bagian E6 — Jalankan cut patches per-kelas (FASE 2, jalankan SETELAH semua task E3 COMPLETED) ===
from forestwatch.data.patches import list_patches

manifest_eval = {'source': 'transfer_eval_gel4', 'classes': {}}
for slug in EVAL_REGIONS:
    tile_dir  = TILES_TRANSFER_EVAL / slug
    patch_dir = PATCHES_TRANSFER_EVAL / slug
    n_tif = len(sorted(tile_dir.glob('*.tif'))) if tile_dir.exists() else 0
    if n_tif == 0:
        print(f'  [skip] {slug}: belum ada .tif di {tile_dir} (task E3 selesai?).')
        continue
    cut_patches_resilient(tile_dir, patch_dir,
                          patch_size=cfg['patches']['size'], stride=cfg['patches']['size'])
    n_patch = len(list_patches(patch_dir))
    manifest_eval['classes'][slug] = {'n_tif': int(n_tif), 'n_patch': int(n_patch)}
    print(f'  [ok] {slug}: {n_tif} tif -> {n_patch} patch')

save_json(manifest_eval, DIST_DIR / 'transfer_eval_export_manifest.json')
print('FASE 2 (cut) selesai. Manifest -> Distribution_Reports/transfer_eval_export_manifest.json')


In [ ]:
# === Bagian E7 — Verifikasi AKTUAL pasca-cut: distribusi piksel per kelas (bukan estimasi lagi) ===
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm
from forestwatch.constants import CLASS_NAMES, N_CLASSES

TARGET_ADD_PER_CLASS = {2: 100_000_000, 3: 100_000_000, 4: 100_000_000, 5: 50_000_000, 6: 100_000_000}
eval_extra_files = list_patches(PATCHES_TRANSFER_EVAL)
print(f'Total patch gelombang-4 (semua kelas gabung): {len(eval_extra_files)}')


def _read_lab(f):
    return np.load(f)['lab']


dist_eval = {c: 0 for c in range(N_CLASSES)}
with ThreadPoolExecutor(max_workers=32) as exe:
    for lab in tqdm(exe.map(_read_lab, eval_extra_files), total=len(eval_extra_files),
                    desc='Distribusi gel.4 eval-only'):
        u, cnt = np.unique(lab, return_counts=True)
        for cls, n in zip(u.tolist(), cnt.tolist()):
            if 0 <= int(cls) < N_CLASSES:
                dist_eval[int(cls)] += int(n)

print('\nDistribusi piksel TRANSFER GELOMBANG-4 (gabungan, AKTUAL pasca-cut) vs target per-kelas '
      '(100 jt / Tambang 50 jt):')
for c in range(N_CLASSES):
    tgt = TARGET_ADD_PER_CLASS.get(c)
    pct = f'{100 * dist_eval[c] / tgt:5.1f}%' if tgt else ' n/a '
    print(f'  {CLASS_NAMES[c]:<16}: {dist_eval[c]:>12,} px  ({pct:>6} target)')

save_json({'counts': {str(c): dist_eval[c] for c in range(N_CLASSES)},
           'target_per_class': TARGET_ADD_PER_CLASS,
           'n_patch': len(eval_extra_files)},
          DIST_DIR / 'distribution_gel4_eval.json')
print('\nVerifikasi "memang ada di 5 kelas itu" (per piksel, bukan cuma estimasi) -- kelas 2-6')
print('di tabel atas semua HARUS > 0 sebelum lanjut ke E8:')
for c in (2, 3, 4, 5, 6):
    assert dist_eval[c] > 0, (
        f'{CLASS_NAMES[c]} = 0 piksel di gelombang-4 -- region utk kelas ini gagal, revisi E1/E2.')
print('OK -- kelima kelas (2-6) terbukti punya piksel NYATA (terhitung dari .npz hasil cut, '
      'bukan estimasi) di gelombang-4. Aman dipakai sbg eval-only -- tidak diturunkan dari\n'
      'train manapun (region genuinely baru, lihat cross-check E1).')


In [ ]:
# === Bagian E8 — Split FAIR & merata (train_new/val_new/test_new), reuse split_files() ===
# Rasio SAMA dgn konvensi proyek (80/10/10, split_files seed=42) -- konsisten, bukan ad-hoc.
# PENTING: train_new TIDAK dipakai utk fine-tuning model_1 (leakage, lihat markdown atas) --
# tersimpan di manifest sbg aset utk skenario retrain-dari-nol di masa depan. improve_model.ipynb
# HANYA mengonsumsi val_new + test_new.
from forestwatch.data.dataset import split_files

train_new, val_new, test_new = split_files(eval_extra_files, train_ratio=0.8, val_ratio=0.1, seed=42)


def _rel(files):
    return ['/'.join(Path(f).parts[-3:]) for f in files]


save_json({
    'note': ('Split gelombang-4 eval-only. train_new TIDAK dipakai utk fine-tuning model_1 '
             '(leakage) -- hanya val_new+test_new yg dikonsumsi improve_model.ipynb.'),
    'base_dir': str(PATCHES_TRANSFER_EVAL),
    'n_train_new': len(train_new), 'n_val_new': len(val_new), 'n_test_new': len(test_new),
    'train_new': _rel(train_new), 'val_new': _rel(val_new), 'test_new': _rel(test_new),
}, DIST_DIR / 'split_manifest_gel4_eval.json')

print(f'Split gelombang-4: train_new={len(train_new)} (TIDAK dipakai improve_model.ipynb), '
      f'val_new={len(val_new)}, test_new={len(test_new)}')
print('Manifest -> Distribution_Reports/split_manifest_gel4_eval.json')
print('\nLangkah selanjutnya: di improve_model.ipynb, load manifest ini, gabungkan')
print('val_new+test_new ke val_p/test_p Papua-only yang sudah ada (lihat plan).')
